In [ ]:
import tensorflow as tf
from keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, Dropout, Flatten, Dense
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
import h5py
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
import glob
import os

# GPU setup (optional)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)



In [ ]:
data_folder = '/scratch/projects/MADS2025-CiJR/rugby_brain_strain_CNN/data/cghs'

# Make sure the path exists
if not os.path.exists(data_folder):
    print("Folder does not exist:", data_folder)
else:
    # Recursive search for all .h5 files
    file_list = glob.glob(os.path.join(data_folder, '**', '*.h5'), recursive=True)
    print(f"Found {len(file_list)} files:")
    for f in file_list:
        print(f)

X = []
y = []

for file_path in file_list:
    with h5py.File(file_path, 'r') as hf:
        for key in hf.keys():
            group = hf[key]
            if 'impact_location' not in group.attrs:
                continue
            impact_loc = group.attrs['impact_location']

            perm_name = 'perm_xyz'
            lin_name = 'lin_perm_xyz'

            if perm_name not in group or lin_name not in group:
                continue

            rot = group[perm_name][:]
            lin = group[lin_name][:]

            if rot.shape != lin.shape or rot.ndim != 3 or rot.shape[0] != 1:
                continue

            # Convert (1, 3, L) -> (3, L, 1) and stack as channels
            rot = rot.transpose(1, 2, 0)
            lin = lin.transpose(1, 2, 0)
            sample = np.concatenate([rot, lin], axis=2)  # (3, L, 2)

            X.append(sample)
            y.append(impact_loc)

# Convert to numpy arrays
if len(X) > 0:
    X = np.stack(X, axis=0)
    y = np.stack(y, axis=0)
else:
    X = np.array([])
    y = np.array([])

# 80:20 train-test split
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
        stratify=np.argmax(y, axis=1)
    )

    input_shape = X_train.shape[1:]
    num_classes = y_train.shape[1]

    print(f"Total samples: {len(X)}")
    print(f"Train samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

else:
    X_train, X_test = [], []
    y_train, y_test = [], []
    input_shape = (3, 1000, 2)
    num_classes = 29

best_params = None



In [ ]:
# 10-fold CV to tune filters/kernel/stride using PR-AUC (best for imbalance)

def build_model(params, input_shape, num_classes):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel1"],
            strides=params["stride1"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel2"],
            strides=params["stride2"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel3"],
            strides=params["stride3"],
            activation='relu',
            padding='valid'
        ),
        Dropout(params.get("dropout", 0.2)),
        Flatten(),
        Dense(units=10, activation='relu'),
        Dense(units=num_classes, activation='sigmoid')
    ])

    optimizer = Adam(learning_rate=params.get("lr", 1e-4))
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(curve="PR", multi_label=True, num_labels=num_classes, name="pr_auc"),
            tf.keras.metrics.AUC(curve="ROC", multi_label=True, num_labels=num_classes, name="roc_auc"),
        ]
    )
    return model

param_grid = [
    {
        "filters": 16,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
    },
    {
        "filters": 32,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
    },
    {
        "filters": 32,
        "kernel1": (3, 7),
        "stride1": (1, 2),
        "kernel2": (1, 7),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
    },
]

if len(X_train) > 0:
    strat_labels = np.argmax(y_train, axis=1)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_results = []

    for params in param_grid:
        fold_scores = []
        for tr_idx, val_idx in skf.split(X_train, strat_labels):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            model = build_model(params, input_shape=X_train.shape[1:], num_classes=y_train.shape[1])

            early_stopping = EarlyStopping(
                monitor='val_pr_auc',
                patience=8,
                mode='max',
                restore_best_weights=True
            )

            model.fit(
                X_tr,
                y_tr,
                epochs=60,
                batch_size=64,
                validation_data=(X_val, y_val),
                callbacks=[early_stopping],
                verbose=0
            )

            y_pred = model.predict(X_val, verbose=0)
            pr_auc = average_precision_score(y_val, y_pred, average='macro')
            fold_scores.append(pr_auc)

        mean_pr_auc = float(np.mean(fold_scores))
        cv_results.append((mean_pr_auc, params))
        print(f"Params {params} -> mean PR-AUC: {mean_pr_auc:.4f}")

    best_pr_auc, best_params = max(cv_results, key=lambda x: x[0])
    print(f"Best params: {best_params}")
    print(f"Best mean PR-AUC: {best_pr_auc:.4f}")
else:
    best_params = None



In [ ]:
input_shape = X_train.shape[1:] if len(X_train) else input_shape
num_classes = y_train.shape[1] if len(X_train) else num_classes

default_params = {
    "filters": 32,
    "kernel1": (3, 10),
    "stride1": (1, 2),
    "kernel2": (1, 10),
    "stride2": (1, 2),
    "kernel3": (1, 5),
    "stride3": (1, 1),
}

if "best_params" in globals() and best_params:
    model = build_model(best_params, input_shape, num_classes)
else:
    model = build_model(default_params, input_shape, num_classes)

model.summary()



In [ ]:
# Early stopping callback (validation-based)
early_stopping = EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=20,
    restore_best_weights=True
)

# Model training
history = model.fit(
    X_train,
    y_train,
    epochs=250,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    shuffle=True
)



In [ ]:
model.save('cnn_location.keras')


In [ ]:
loaded_model = tf.keras.saving.load_model("cnn_location.keras")
